# سبق 13 - کوگنی نالج گراف کے ساتھ ایجنٹ میموری


## سیٹ اپ

یہ نوٹ بک دکھاتی ہے کہ کیسے ایک ذہین **کوڈنگ اسسٹنٹ** بنایا جائے جو مستقل میموری کے ساتھ [**Cognee**](https://www.cognee.ai/) نالج گرافس اور **Microsoft Agent Framework** (MAF) کا استعمال کرتا ہے۔

Cognee غیر منظم متن کو ایک منظم، استفساری نالج گراف میں تبدیل کرتا ہے جس کی پشت پناہی ویکٹر ایمبیڈنگز کرتی ہے — جو آپ کے ایجنٹ کو ایک معیاری، تعلقات سے باخبر طویل مدتی میموری فراہم کرتا ہے۔

### آپ کیا سیکھیں گے
1. **نالج گرافس بنائیں**: ڈویلپر پروفائلز اور بہترین طریقوں کو منظم، استفساری علم میں تبدیل کریں۔
2. **Cognee کو MAF کے ساتھ مربوط کریں**: `@tool` افعال کا استعمال کریں تاکہ ایک MAF ایجنٹ Cognee کے نالج گراف سے استفسار کرسکے۔
3. **سیشن سے آگاہ گفتگو**: ایک ہی سیشن میں متعدد سوالات کے درمیان سیاق و سباق کو برقرار رکھیں۔
4. **طویل مدتی میموری**: اہم علم کو سیشنز کے درمیان محفوظ کریں اور نئی گفتگو میں اسے بازیافت کریں۔

### پیشگی ضروریات
- Python 3.9+
- Redis مقامی طور پر چل رہا ہو (`docker run -d -p 6379:6379 redis`) سیشن مینجمنٹ کے لیے
- ایک LLM API کلید (مثلاً OpenAI) — `.env` میں `LLM_API_KEY` سیٹ کریں
- `.env` میں `CACHING=true` (Cognee سیشنز کے لیے ضروری)
- ایک Azure AI Foundry پروجیکٹ جس میں ایک تعین شدہ چیٹ ماڈل ہو
- `.env` میں `AZURE_AI_PROJECT_ENDPOINT` اور `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI میں تصدیق شدہ (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ایجنٹ کی یادداشت کی اقسام

یہ نوٹ بک مین لیسن 13 نوٹ بک سے وہی تین یادداشت کی اقسام دریافت کرتی ہے، لیکن طویل مدتی یادداشت کے طور پر Cognee کو استعمال کرتی ہے:

| یادداشت کی قسم | طریقہ کار | مدت زندگی |
|---|---|---|
| **ورکنگ** | `agent.create_session()` (MAF) | واحد گفتگو |
| **مختصر مدتی** | Cognee سیشن کیش (Redis) | واحد سیشن |
| **طویل مدتی** | Cognee نالج گراف + ویکٹرز | مستقل |

### Cognee کی یادداشت کا فن تعمیر
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## کوگنی اسٹوریج تیار کریں


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## حصہ 1 — علم کا بنیادی ڈھانچہ بنانا

ہم اپنے کوڈنگ اسسٹنٹ کے لیے ایک جامع علم کا بنیادی ڈھانچہ بنانے کے لیے تین قسم کے ڈیٹا کو شامل کرتے ہیں:

1. **ڈیولپر پروفائل** — ذاتی مہارت اور تکنیکی پس منظر  
2. **پائتھن بہترین طریقے** — پائتھن کا ذن عملی رہنما اصولوں کے ساتھ  
3. **تاریخی گفتگوئیں** — ماضی کے سوال و جواب کے سیشنز جو ڈیولپرز اور اے آئی اسسٹنٹس کے درمیان ہوئے


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## علم کے گراف کا تصور کریں

Cognee وہ اکائیاں اور تعلقات جو اس نے نکالے ہیں، ان کی ایک تعاملی HTML بصری تصویر پیش کر سکتا ہے۔


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## میموری کو میمیفائی کے ساتھ بھرپور بنائیں

`memify()` نالج گراف کا تجزیہ کرتا ہے اور ذہین قواعد تیار کرتا ہے — پیٹرنز، بہترین طریقہ کار، اور تصورات کے درمیان تعلقات کی نشاندہی کرتا ہے۔


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## حصہ 2 — Cognee ٹولز کے ساتھ MAF ایجنٹ

اب ہم ایک MAF ایجنٹ بناتے ہیں جو `@tool` فنکشنز کے ذریعے Cognee کے نالج گراف سے سوال کر سکتا ہے۔ اس سے ایجنٹ کو گراف سے آگاہ سیمنیٹک سرچ کی پوری طاقت استعمال کرنے کی اجازت ملتی ہے جبکہ سیشنز کے ذریعے بات چیت کے سیاق و سباق کو برقرار رکھا جاتا ہے۔


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## ورکنگ میموری وج سیشنز

`AgentSession` (جو `agent.create_session()` کے ذریعے بنائی جاتی ہے) ایک سیشن کے اندر ورکنگ میموری فراہم کرتی ہے۔ ایجنٹ پہلے کے پیغامات کی طرف رجوع کر سکتا ہے جبکہ Cognee کے طویل مدتی نالج گراف سے بھی سوالات کر سکتا ہے۔


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## نیا سیشن — طویل مدتی یادداشت برقرار رہتی ہے

نیا سیشن شروع کرنے سے ورکنگ میموری صاف ہو جاتی ہے، لیکن Cognee نالج گراف ابھی بھی دستیاب ہے۔ ایجنٹ ایک بالکل نئی گفتگو میں اسی طویل مدتی معلومات کو واپس حاصل کر سکتا ہے۔


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## خلاصہ

اس نوٹ بک میں آپ نے ایک کوڈنگ اسسٹنٹ بنایا ہے جو **MAF کی ورکنگ میموری** (`agent.create_session()`) کو **Cognee کے طویل مدتی نالج گراف** کے ساتھ جوڑتا ہے۔

### آپ نے کیا سیکھا
1. **نالج گراف کی تعمیر**: Cognee غیر ساختہ متن کو لیتا ہے اور ایک گراف + ویکٹر میموری بناتا ہے۔
2. **memify کے ساتھ گراف کی افزودگی**: آپ کے موجودہ گراف کے اوپر اخذ شدہ حقائق اور زیادہ گہرے تعلقات۔
3. **MAF + Cognee انضمام**: `@tool` افعال MAF ایجنٹ کو Cognee کے گراف سے قدرتی طور پر سوال پوچھنے دیتے ہیں۔
4. **ورکنگ میموری + طویل مدتی میموری**: `AgentSession` (کی سہولت `agent.create_session()`) سیشن کا سیاق و سباق فراہم کرتا ہے جب کہ Cognee مستقل علم دیتا ہے۔
5. **NodeSets کے ساتھ فلٹر شدہ تلاش**: نالج گراف کے مخصوص ذیلی حصوں کو ہدف بنایا جا سکتا ہے (مثلاً صرف اصول)۔

### اہم نکات
- **Cognee** خام متن کو ایک ساختہ، تعلقاتی واقف میموری میں تبدیل کرتا ہے — جو ایک سیدھی ویکٹر اسٹور سے زیادہ طاقتور ہے۔
- **`@tool` افعال** MAF ایجنٹس اور بیرونی علم کے نظاموں کو صاف طریقے سے جوڑ دیتے ہیں۔
- **`AgentSession`** (کی سہولت `agent.create_session()`) ہر گفتگو کے سیاق و سباق کو طویل مدتی علم سے جدا رکھتا ہے۔
- وہی نالج گراف متعدد سیشنز اور ایجنٹس کے لیے کام کرتا ہے۔

### عملی استعمالات
- **ڈویلپر کوپائلٹس**: کوڈ ریویو، واقعہ تجزیہ، معماری اسسٹنٹس
- **کسٹمر-سامنا کوپائلٹس**: مصنوعات کی دستاویزات، FAQs، اور CRM نوٹس کے اوپر سپورٹ ایجنٹس
- **اندرونی ماہر کوپائلٹس**: پالیسی، قانونی، یا سیکورٹی اسسٹنٹس جو رہنما اصولوں پر دلیل کرتے ہیں
- **متحدہ ڈیٹا پرتیں**: ساختہ اور غیر ساختہ ڈیٹا کو ایک سوالیہ گراف میں یکجا کرنا

### اگلے اقدامات
- Cognee میں زمانی آگاہی کے ساتھ تجربہ کریں
- مخصوص دائرہ کار کے گراف معیار کے لیے OWL آنٹولوجی کی تعریف کریں
- وقت کے ساتھ بازیابی کو بہتر بنانے کے لیے صارف کی رائے کے لوپ شامل کریں
- ایک ہی Cognee میموری پرت کو شیئر کرنے والے کثیر ایجنٹ نظاموں پر اسکیل کریں


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**دستبرداری**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ اگرچہ ہم درستگی کی کوشش کرتے ہیں، براہ کرم یہ بات ذہن میں رکھیں کہ خودکار ترجمے میں غلطیاں یا بے دقتیاں ہو سکتی ہیں۔ اصل دستاویز اپنی مادری زبان میں معتبر ذریعہ سمجھی جانی چاہیے۔ اہم معلومات کے لیے پیشہ ور انسانی ترجمہ تجویز کیا جاتا ہے۔ ہم اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کے لیے ذمہ دار نہیں ہیں۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
